# Lab 1: Logical Agents with Propositional Logic

This lab consists of two parts:

- **Part 1:** Implement propositional logic, evaluate sentences in a given model, and query a knowledge base.
- **Part 2:** Use the propositional logic and query function from Part 1 to create a knowledge-based agent that traverses the Wumpus World.

Part 2 builds directly on Part 1, so make sure Part 1 is implemented and tested before you start Part 2.


# Part 1: Making Propositional Logic (PL) and Querying PL Knowledge base
In this part you will implement propositional logic which will be the foundation of our AI agent.
Initially it might seem strange to implement propositional logic, since Python already has it implemented in the simple ```not```, ```and```, ```or```, and ```==``` operators (implication can be derived from these as well).
While it's true that any propositional logic statement can be evaluated using Python, these statements cannot be stored in their unevaluated form (which is needed to form a Knowledge Base (KB)).
Additionally, we cannot create variables without assigning them a value in standard Python, but when we later want to iterate over all possible models the variables will need to take on many possible values.

So we will need to implement our own version of propositional logic in Python.
For variables we will use strings to represent them and later we can use dictionaries to represent models and then look up the value of each variable in that model.
For True and False we will simply use the existing ```True``` and ```False``` values in Python.
For all the operators we will implement classes that contain the logic sentences they operate on.

> **Task 1:** Given a skeleton for the "and" operator, you need to expand it to each of the other operators.

> **Task 2:** Extend the operators so that they can take more than two sentences, e.g. ```And(s1,s2,s3, ...)```. Also implement a function that, given a binary operator and a list of sentences, chains them together (e.g. ```f(And, [s1, s2, s3]) -> And(s1, And(s2, s3))```).

For some operators this isn’t necessary or doesn't make sense. For example, ```Not``` applies to only one sentence. Implication is not associative, so ```(p→q)→r``` is not equivalent to ```p→(q→r)```. 

In [9]:
class And:
    def __init__(self, *args):
        if len(args) < 2:
            raise ValueError("And operator requires at least two arguments.")
        self.args = args
        
        
    def __repr__(self):
        return f"({' ∧ '.join(str(arg) for arg in self.args)})"

class Or:
    def __init__(self, *args):
        if len(args) < 2:
            raise ValueError("Or operator requires at least two arguments.")
        self.args = args

    def __repr__(self):
        return f"({' ∨ '.join(str(arg) for arg in self.args)})"

# (Equivalent <=>)
class Equals:
    def __init__(self, A, B):
        self.args = (A, B)

    def __repr__(self):
        return f"({self.args[0]} \u21d4 {self.args[1]})"

class Implies:
    def __init__(self, A, B):
        self.args = (A, B)

    def __repr__(self):
        return f"({self.args[0]} \u21d2 {self.args[1]})"

class Not:
    def __init__(self, A):
        self.args = (A,)

    def __repr__(self):
        return f"(\u00ac{self.args[0]})"

In [10]:
def chain(operator, sentences):
    '''
    Chains a list of sentences together with a binary operator, nesting to the right.
    Example: chain(And, [s1, s2, s3]) -> And(s1, And(s2, s3))
    Args:
        operator (class): The operator class to use, e.g. And, Or, Equals, Implies
        sentences ([any]): The sentences to chain, at least one
    Returns (any): The chained sentence
    '''
    sentences = list(sentences)
    if len(sentences) == 0:
        raise ValueError("chain needs at least one sentence")
    if len(sentences) == 1:
        return sentences[0]
    return operator(sentences[0], chain(operator, sentences[1:]))


In [11]:
s = Implies(And('rain', Not('sunny')), Equals('umbrella', True))
print(s)
print(s.args)      
KB = {Not('rain'), Implies('rain', 'umbrella')}
print(len(KB))
print(KB)


((rain ∧ (¬sunny)) ⇒ (umbrella ⇔ True))
((rain ∧ (¬sunny)), (umbrella ⇔ True))
2
{(¬rain), (rain ⇒ umbrella)}


In [12]:
print(And('sunny', 'rain', 'snow'))
print(Or('rain', 'cloudy', 'storm'))
print(And('sunny', 'rain', 'snow').args)
print(chain(And, ['s1', 's2', 's3']))
print(chain(Implies, ['p', 'q', 'r']))
print(chain(Or, ['only_one']))
s = chain(And, ['s1', 's2', 's3'])
print(type(s).__name__, s.args)
And('a')   # raises ValueError


(sunny ∧ rain ∧ snow)
(rain ∨ cloudy ∨ storm)
('sunny', 'rain', 'snow')
(s1 ∧ (s2 ∧ s3))
(p ⇒ (q ⇒ r))
only_one
And ('s1', (s2 ∧ s3))


ValueError: And operator requires at least two arguments.

## Evaluating propositional sentences in a given model

Being able to form sentences in propositional logic is not very useful if we cannot actually evaluate them. A start of the function can be found below, but it is missing most things. 

> **Task 3:** Implement a function ```evaluate``` that takes a sentence in propositional logic and a dictionary with the truth values of the different variables in a given model and returns the truth value of the sentence in that model.

> **Task 4:** Extend ```evaluate``` to handle unknown values (```None```): it should return ```True``` if the sentence is true, ```False``` if it is false, and ```None``` if the answer cannot be known because some variables are unknown.

Note that the function for replacing the variables with their correct truth-value will currently give an error if a variable is missing, but we want it to return ```None``` to represent an unknown value instead.

This is done to make evaluation of large models more efficient down the line.
We add the concept of an unknown or uninteresting truth value using the Python ```None```-type.
The idea with unknown values is that instead of iterating over every value for every variable to evaluate a model, we can decide to only iterate over some variables and set the others as unknown.
However if an important variable is set to unknown we might not be able to get the correct answer.
In order to protect against this the ```evaluate``` function will return ```True``` if the sentence is true, ```False``` if its false, and ```None``` if we cannot know since some variables are unknown to us.
In order to deal with the ```None``` values you will have to think what the answer to many different operators should be. For example, we know that ```And(False, None)``` should be ```False``` since ```And``` always returns ```False``` when at least one sub-sentence is ```False```.
You will also need to experiment with Python to see if the Python operators give the correct behavior. Does ```False == None``` return ```None```?
If not, how should you deal with it?


In [14]:
def evaluate(sentence, variables):
    '''
    Evaluates the value of a logic sentence using the values given to the variables in a dictionary.
    Args:
        sentence (any): A boolean, None, a variable string, or an operator class containing subsentences
        variables (dict): A dictionary mapping variables represented by strings to boolean values
    Returns (bool/None): The truth value of the sentence with the given variables or None if the answer is unknown
    '''
    # Constants (True/False) and unknowns (None) are already truth values
    if sentence is None or isinstance(sentence, bool):
        return sentence

    # A variable: look up its value in the model; a missing variable is unknown (None)
    if isinstance(sentence, str):
        return variables.get(sentence)

    # Anything else must be one of our operators
    if not isinstance(sentence, (Not, And, Or, Implies, Equals)):
        raise TypeError(f"Cannot evaluate {sentence!r}")

    # Evaluate every sub-sentence first (recursion), then combine the results
    values = [evaluate(sub, variables) for sub in sentence.args]

    if isinstance(sentence, Not):
        return not values[0]
    elif isinstance(sentence, And):
        return all(values)
    elif isinstance(sentence, Or):
        return any(values)
    elif isinstance(sentence, Implies):
        return (not values[0]) or values[1]
    elif isinstance(sentence, Equals):
        return values[0] == values[1]


In [15]:
model = {'rain': True, 'sunny': False, 'umbrella': True}
print(evaluate(True, model))
print(evaluate('rain', model))
print(evaluate('play', model))
print(evaluate(Equals('rain', Not('sunny')), model))
print(evaluate(Implies('rain', 'umbrella'), model))
print(evaluate(Equals(Implies(Not('sunny'), Not('umbrella')), False), model))
print(evaluate(And('sunny', 'rain', 'umbrella'), model))
print(evaluate(Or('sunny', 'rain', 'umbrella'), model))
print(evaluate(chain(And, ['rain', 'umbrella', Not('sunny')]), model))


True
True
None
True
True
True
False
True
True


### Testing the evaluate function

> **Task 5:** Implement and evaluate the sentences of an example below for its test model, and check that you get the expected results.

We have now created a propositional logic grammar and a function to evaluate the truth value of sentences for a given model.
In order to make sure everything is correct we introduce a testing space below and ask you to implement and evaluate a number of sentences for a given model in order to see that you get the correct result.

Below you are given a model and are then supposed to implement and run evaluation on the sentences below, with the given expected results:

#### Example 1
- $rain \leftrightarrow \neg sunny$ is ```True```
- $rain \rightarrow umbrella$ is ```True```
- $(\neg sunny \rightarrow \neg umbrella) \leftrightarrow False$ is ```True```
- $(sunny \vee rain) \rightarrow play$ is ```None```
- $sunny \rightarrow play$ is ```True```
- $\neg rain \leftrightarrow play$ is ```None```
- $\neg play$ is ```None```
- $(sunny \wedge rain \wedge snow) \leftrightarrow False$ is ```True```

#### Example 2
- $rain \leftrightarrow \neg sunny$ is ```True```
- $umbrella \leftrightarrow rain$ is ```True```
- $\neg cloudy \leftrightarrow False$ is ```True```
- $(sunny \vee rain) \rightarrow storm$ is ```None```
- $\neg umbrella \leftrightarrow False$ is ```True```
- $sunny \rightarrow cloudy$ is ```True```
- $(rain \wedge cloudy) \rightarrow storm$ is ```None```
- $(rain \vee cloudy \vee storm)$ is ```True```

#### Example 3
- $windy \leftrightarrow \neg humid$ is ```True```
- $power \rightarrow alert$ is ```False```
- $(\neg humid \rightarrow \neg power) \leftrightarrow False$ is ```True```
- $(windy \vee humid) \rightarrow generator$ is ```None```
- $alert \rightarrow windy$ is ```True```
- $(\neg alert) \leftrightarrow False$ is ```False```
- $\neg generator$ is ```None```
- $(windy \wedge humid \wedge alert) \leftrightarrow False$ is ```True```

The test model for each example is given below. You are free to test with any (or all) of the examples.

In [4]:
# test_model = {'rain': True,'sunny': False,'umbrella': True,'snow': False,} # Example 1
# test_model = {'rain': True,'sunny': False,'umbrella': True,'cloudy': True,} # Example 2
# test_model = {'windy': True,'humid': False,'power': True,'alert': False,} # Example 3


## Querying for possible models from a Knowledge Base

> **Task 6:** Implement the ```query``` function (skeleton below). Given a query sentence, a knowledge base, and a list (or set) of relevant variables, it returns the "valid" models where the query is ```True```, ```False```, or ```None```.

Now we are ready to create a knowledge base.
Making a knowledge base is not that difficult, it's simply a collection of logical sentences that are known (or presumed) to be true in a given environment.

To create this collection we will use the Python ```set``` data structure.
The reason is that sets can only contain one of each element so we don't risk adding the same sentence multiple times.
Sets can be created by listing a number of elements in curly brackets (e.g. ```{s1, s2}```), and elements can be added and removed with the ```.add()``` and ```.remove()``` functions.
Note that an empty set must be created with ```set()```, since ```{}``` creates an empty dictionary.

*Hint*: For a set to recognize that two sentences are the same (e.g. two separately created ```Not("rain")```), your operator classes need ```__eq__``` and ```__hash__``` methods. Without them every new object counts as a different sentence, and ```.remove()``` will not find it.

The more difficult part is then creating a function for querying the knowledge base for whether a new sentence is true or not.
One simple way to do this query is to test every possible model (every possible combination of truth values for the variables).
For each model we figure out whether the query sentence is True or False, we then also check that all sentences in the knowledge base are True for that model.
Since we "know" that each sentence in the knowledge base should be true, any model that doesn't align with this can be discarded as an "invalid" model of our environment.
For all "valid" models we then return whether the query is True or False in that model.

Of course many environments have a huge amount of variables which would then require us to check a huge number of models.
If we have $n$ variables in the environment we would need to check each sentence $2^n$ times.
However, not every variable is necessarily relevant for our query sentence.

Using our knowledge of the environment we might be able to deduce the minimum number of variables that are relevant for our query and only test all combinations of them.
This is why we implemented the concept of unknown (```None```) values earlier.
Due to this we also consider a model "valid" if a sentence in the knowledge base evaluates as unknown.
This comes with a risk of course: if we mistakenly exclude a relevant variable in our query a model that should be "invalid" might be considered "valid" since a False evaluation might now be unknown.
Additionally, we need to return all "valid" models where the query is unknown.

We can then use this query function to figure out things about the environment.
If a query sentence only gives "valid" models in which it is True, then we know the sentence is True in the environment.
Conversely, if all returned models are False, the sentence is False in the environment.
If it returns some True models and some False models we know that, given what we know of the environment, we cannot yet tell whether the sentence is True or not (in some advanced scenarios we might be able to use the number of True/False models to estimate the likelihood of the query being True/False in the actual environment).
If some of the returned models are unknown models, we know that there are relevant variables that were not considered in the query.
This likely means we have an error in the code that determines which variables should be relevant for a query.
Note that many variables that are not in the query sentence could still be relevant.

Finally, if the function returns no valid models at all, this means that there are contradictions in the knowledge base and something has gone wrong in its construction.

Below you will find the skeleton of the query function for you to complete.

In [5]:
import itertools
# Hint: You can use the ```itertools.product()``` function to generate a list of all possible combinations of a list 
# [1,2,3]*3 creates a list where the elements of the list are repeated 3 times [1,2,3,1,2,3,1,2,3]
# function(*[1,2,3]) is the same as function(1, 2, 3)
list(itertools.product(*[(1,2,3)]*3))

[(1, 1, 1),
 (1, 1, 2),
 (1, 1, 3),
 (1, 2, 1),
 (1, 2, 2),
 (1, 2, 3),
 (1, 3, 1),
 (1, 3, 2),
 (1, 3, 3),
 (2, 1, 1),
 (2, 1, 2),
 (2, 1, 3),
 (2, 2, 1),
 (2, 2, 2),
 (2, 2, 3),
 (2, 3, 1),
 (2, 3, 2),
 (2, 3, 3),
 (3, 1, 1),
 (3, 1, 2),
 (3, 1, 3),
 (3, 2, 1),
 (3, 2, 2),
 (3, 2, 3),
 (3, 3, 1),
 (3, 3, 2),
 (3, 3, 3)]

In [6]:
def query(query, KB, variables):
    '''
    For a given query and knowledge base, returns the models where all KB sentences are true
    in three lists depending on if the query is True, False, or None
    Args:
        query (any): Logic sentence to check the truth value of
        KB ({any}): A set of logic sentences, none of which should be False for the returned models
        variables ([str]): A list (or set) of strings that make up the relevant variables for the query
    Returns ([{str:bool}],[{str:bool}],[{str:bool}]): Lists of the "valid" models where the query was True, False, or None
    '''

### Testing the querying function

> **Task 7:** Create a knowledge base from the sentences you used to test ```evaluate```, query it with the sentences of the same example below, and check that you get the expected results.

No complex function is complete without testing.
To do this we will use the sentences from testing our evaluation function as the knowledge base.
You will be given a list of "relevant" variables and then you will query the knowledge base with the following sentences with the given expected results.

The query function must follow the stated design requirements (argument and return types).

The solution is accepted only if the querying function exactly returns the stated expected result of each query.

#### Example 1
- $True$ returns 2 True models (and 0 False/None models).
- $rain \leftrightarrow \neg sunny$ returns 2 True models.
- $play \wedge cloudy$ returns 1 False model, and 1 unknown model.
- $sunny \leftrightarrow play$ returns 2 unknown models.
- $(sunny \leftrightarrow \neg cloudy) \leftrightarrow False$ returns 1 True and 1 False model.

#### Example 2
- $True$ returns 1 True model (and 0 False/None models).
- $rain \leftrightarrow \neg sunny$ returns 1 True model.
- $storm \wedge cloudy$ returns 1 unknown model.
- $sunny \leftrightarrow storm$ returns 1 unknown model.
- $(sunny \leftrightarrow \neg cloudy) \leftrightarrow False$ returns 1 False model.

#### Example 3
- $True$ returns 1 True model (and 0 False/None models).
- $windy \leftrightarrow \neg humid$ returns 1 True model.
- $power \wedge generator$ returns 1 unknown model.
- $alert \leftrightarrow generator$ returns 1 unknown model.
- $(alert \leftrightarrow \neg humid) \leftrightarrow False$ returns 1 False model.

The test variables for each example are given below. You are free to test with any (or all) of the examples.

In [7]:
# test_variables = {"rain", "umbrella", "cloudy", "sunny"} # Example 1
# test_variables = {"rain", "sunny", "umbrella", "cloudy"} # Example 2
# test_variables = {"windy", "humid", "power", "alert"} # Example 3


KB = set() #Add all the sentences that were used to test evaluation previously to the knowledge base



# Part 2: Living in the Wumpus World: Logical Agents with propositional logic

In this part we will learn how to use propositional logic and knowledge bases, which you created in Part 1, to create a simple agent to traverse the Wumpus World.

This part includes:

- Implementing an agent that keeps a knowledge base of the Wumpus World and queries it for which tiles are safe to visit.

The content of this lab relates strongly to the material of Chapter 6 (Logical Agents) in the course book (Artificial Intelligence: A Modern Approach).
The environment we will tackle in this lab is called the Wumpus World and is covered in the course book as well (Chapter 6.2)

## Implementation of the Wumpus World

Below you will find a code implementation of the Wumpus World along with a few helpful definitions and functions.
The Wumpus World is fully implemented since this part is about building a propositional logic knowledge base system to build an AI around.
However, since the AI will interact with the Wumpus World, being familiar with the code and functions will help you use them later.

In [8]:
#Imports
import random
import itertools

# Directions
RIGHT = 0
DOWN  = 1
LEFT  = 2
UP    = 3

# How much the agent will move if it takes a step forward for each direction
direction_step = [(1,0), (0,-1), (-1,0), (0,1)]

# Objects
AGENT   = "a"
WUMPUS  = "w"
PIT     = "p"
GOLD    = "g"

# Actions
NONE = ""
FORWARD = "forward"
TURN_LEFT = "turn_left"
TURN_RIGHT = "turn_right"
GRAB = "grab"
SHOOT = "shoot"
CLIMB = "climb"

class WumpusWorld:
    
    def __init__(self):
        self.game_over = False
        self.agent_dir = RIGHT
        self.agent_has_arrow = True
        self.agent_has_gold = False
        self.wumpus_screams = False
        self.non_start_squares = list(itertools.product([0,1,2,3],[0,1,2,3]))
        self.non_start_squares.pop(0)
        self.cave = [[[],[],[],[]],[[],[],[],[]],[[],[],[],[]],[[],[],[],[]]]
        self.cave[0][0].append(AGENT)
        self.agent_pos = (0,0)
        self.random_place(WUMPUS)
        self.random_place(GOLD)
        for square in self.non_start_squares:
            if random.random() < 0.2:
                self.cave[square[0]][square[1]].append(PIT)
        self.performance = 0
        self.last_action = NONE
    
    def random_place(self,object):
        '''
        Randomly places an object into a tile other than the start.
        Args:
            object (any): The object (typically WUMPUS or GOLD) to place
        '''
        pos = random.choice(self.non_start_squares)
        self.cave[pos[0]][pos[1]].append(object)
        
    def show(self):
        '''
        Prints the Wumpus World cave and current agent performance.
        '''
        for y in range(3,-1,-1):
            for x in range(0,4):
                [print(o,end="") for o in self.cave[x][y]]
                print("\t",end="")
            print()
        print(f"Performance: {self.performance}")

    def actuate(self, action):
        '''
        If the game is still running, the input action is performed and the game state is moved forward.
        Args:
            action (str): The action the agent should perform
        '''
        # If the game is over, nothing can be done
        if self.game_over:
            return
        
        # store the action that was performed, decrease the performance, and remove scream from memory
        self.last_action = action
        self.performance = self.performance-1
        self.wumpus_screams = False
        
        # If the NONE action is performed, do nothing
        if action == NONE:
            return
        
        # FORWARD action: Move the agent one tile in its current direction.
        # End the game and lower performance if it steps into a PIT or WUMPUS
        if action == FORWARD:
            x, y = self.agent_pos
            self.cave[x][y].remove(AGENT)
            xp, yp = direction_step[self.agent_dir]
            x = min(max(x+xp,0),3)
            y = min(max(y+yp,0),3)
            self.agent_pos = (x,y)
            if WUMPUS in self.cave[x][y] or PIT in self.cave[x][y]:
                self.game_over = True
                self.performance = self.performance-1000
            else:
                self.cave[x][y].append(AGENT)
            return
        
        # TURN_LEFT action: Change the agents direction 90 degrees counter-clockwise
        if action == TURN_LEFT:
            self.agent_dir = (self.agent_dir-1)%4
            return
        # TURN_RIGHT action: Change the agents direction 90 degrees clockwise
        if action == TURN_RIGHT:
            self.agent_dir = (self.agent_dir+1)%4
            return
        # GRAB action: If the agent is in the same tile as the gold, remove it from the cave and add it to the agent
        if action == GRAB:
            x, y = self.agent_pos
            if GOLD in self.cave[x][y]:
                self.agent_has_gold = True
                self.cave[x][y].remove(GOLD)
            return
        # SHOOT action: If the agent has the arrow,
        #    remove it, decrease performance, and, 
        #    if there is a WUMPUS infront of the agent, remove it and add scream perception 
        if action == SHOOT:
            if not self.agent_has_arrow:
                return
            self.agent_has_arrow = False
            self.performance = self.performance-10
            x, y = self.agent_pos
            if self.agent_dir == RIGHT:
                hit_squares = [(min(x+1,3),y),(min(x+2,3),y),(min(x+3,3),y)]
            elif self.agent_dir == DOWN:
                hit_squares = [(x,max(y-1,0)),(x,max(y-2,0)),(x,max(y-3,0))]
            elif self.agent_dir == LEFT:
                hit_squares = [(max(x-1,0),y),(max(x-2,0),y),(max(x-3,0),y)]
            elif self.agent_dir == UP:
                hit_squares = [(x,min(y+1,3)),(x,min(y+2,3)),(x,min(y+3,3))]
            for x, y in hit_squares:
                if WUMPUS in self.cave[x][y]:
                    self.cave[x][y].remove(WUMPUS)
                    self.wumpus_screams = True
            return
        
        # CLIMB action: If the agent is at the starting tile, end the game and add performance if the agent has the gold
        if action == CLIMB:
            if self.agent_pos == (0,0):
                if self.agent_has_gold:
                    self.performance = self.performance+1000
                self.cave[0][0].remove(AGENT)
                self.agent_pos = None
                self.agent_dir = None
                self.game_over = True
            return
    
    def perceive(self):
        '''
        Returns for each percept whether that percept was sensed after the last action.
        '''
        x,y = self.agent_pos
        stench = False
        breeze = False
        for sx,sy in adjacent_tiles(x,y):
            if WUMPUS in self.cave[sx][sy]:
                stench = True
            if PIT in self.cave[sx][sy]:
                breeze = True
        glitter = GOLD in self.cave[x][y]
        bump = False
        if self.last_action == FORWARD:
            if self.agent_dir == RIGHT and x == 3:
                bump = True
            elif self.agent_dir == DOWN and y == 0:
                bump = True
            elif self.agent_dir == LEFT and x == 0:
                bump = True
            elif self.agent_dir == UP and y == 3:
                bump = True
        scream = self.wumpus_screams
        return(stench, breeze, glitter, bump, scream)

def adjacent_tiles(x, y):
    '''
    Helper function that, given a position, returns all tiles that are adjacent to it.
    Args:
        x (int): The x coordinate of the position
        y (int): The y coordinate of the position
    '''
    tiles = []
    for sx,sy in [(x-1,y),(x+1,y),(x,y-1),(x,y+1)]:
        if sx<4 and sx>-1 and sy<4 and sy>-1:
            tiles.append((sx,sy))
    return tiles

## Using logic to understand the Wumpus World

Now that we have a working Propositional Logic, Knowledge Base, and Query system in place, we can finally start using it to create an agent.
Knowledge-based agents can come in many forms, but the most typical behavior is to dump all the relevant knowledge of the world into a knowledge base and then query the knowledge base for what action to perform next.
The percepts from the environment are then used to add new knowledge into the knowledge base to help with the next query.

Doing all of this can require very extensive knowledge bases and complex agents, which is beyond the scope of this assignment.
Instead we will use a very simple procedure for our agent:
- If the agent has the gold we know there must be a safe path back to the entrance, take actions to follow that path back and then use CLIMB to finish the game.
- If there is glitter on the current tile, use the GRAB action to pick it up.
- Otherwise, query every tile for whether it's safe or not.
    - If there is a safe tile that has not yet been visited, calculate a path to it and then follow that path.
        - If there is no path, calculate a path to the next unvisited safe tile.
    - If all safe tiles have been visited or are unreachable, find a path back to the start tile and then use CLIMB to finish the game.

As you can see this agent is very simple, not even considering using the SHOOT action, and treats every tile that it doesn't know for sure to be safe as unsafe.
For now though, this is enough.
Before we can start creating an agent, we need a knowledge base for it to query.
There are many things we know about the Wumpus World, many of which we could turn into logic and put into the knowledge base.
However, the more things we put in the knowledge base the longer the query will take, so we want to stick with only the essentials.
For example, we know that there can only be one WUMPUS and that each tile (except the start) has a 20% chance of being a PIT.
But the 20% chance is difficult to encode in propositional logic and not really relevant to our query, since we only care about things we know for sure.

So what do we put in the knowledge base?
We want to look for safe squares, so we need to put a sentence for each tile defining what it means for that tile to be safe (there is no WUMPUS and no PIT there).
We also need some way to figure out whether there is a WUMPUS on a tile, so we need to add that a tile has stench if and only if there is a WUMPUS on at least one of its adjacent tiles.
The same goes for PIT and breeze.
Of course we also know some things about the starting tile.

Then we need a way to add new knowledge into the knowledge base based on the percepts that are encountered.

So now we need to create two functions.
One that returns the knowledge base about all relevant things we know at the start of each game, and one that given the percepts, the agent's position when those percepts were received, and a knowledge base, adds new relevant knowledge to the knowledge base.
A skeleton for each function is provided below.

> **Task 8:** Implement ```initialWumpusKB()```, which returns a knowledge base with everything known at the start of each game, and ```addWumpusKnowledge()```, which adds new knowledge to the knowledge base based on the agent's percepts and position.

*Hint*: Remember that each sentence needs to be created for each tile and that each tile needs its own versions of all symbols. Try to think of a systematic way to name the symbols based on their tile coordinates.

In [9]:
def initialWumpusKB():
    '''
    Returns a knowledge base of sentences known to be true in all initial Wumpus Worlds
    Returns ({any}): A set of logic sentences
    '''
    # Write a function that creates a set of all relevant starting knowledge.
    return set()

    
def addWumpusKnowledge(percepts, agent_pos, KB):
    '''
    Updates the given knowledge base by adding (and potentially removing) sentences from the knowledge base,
    based on the Wumpus World agent's position and perceptions.
    Args:
        percepts ([bool]): A list of whether each percept is active after the last action
        agent_pos (int, int): The tile currently occupied by the agent
        KB ({any}): The knowledge base of the current Wumpus World to be updated
    '''
    stench, breeze, glitter, bump, scream = percepts
    # Write a function that given percepts adds the relevant knowledge to the KB

## Creating the agent

With knowledge base in hand it's time to create our simple naive agent.

Reading the description of the agent above, you probably noticed that it requires a search algorithm to find a path from one point to another.
Since search algorithms aren't covered in this course a simple implementation of the shortest path algorithm **A\*** has been implemented in the next code block for you to use.
It takes the start tile, goal tile, and a list of all safe tiles and then returns the next tile to walk to.
Even though search-algorithms like **A\*** are not covered in this course, you can read more about them in Chapter 3 (Solving Problems by Searching) of the course book (Artificial Intelligence: A Modern Approach).

You also receive a function to run the game given a class to return actions (an agent), and a skeleton for the agent you will implement based on the earlier description.

> **Task 9:** Implement ```SimpleAgent``` (skeleton below) following the procedure described in *Using logic to understand the Wumpus World*.
>
> **Important:** You must use the ```query()``` function created in Part 1 to solve Part 2.

*Hint*: Querying with every symbol of the Wumpus World as variables means checking an enormous number of models. Think about which few variables are actually relevant for whether a single tile is safe.

In [10]:
# An implementation of a priority queue that is necessary for calculating A*
import heapq

def a_star_on_tiles(start, goal, safe_tiles):
    '''
    Takes a start tile, end tile, and a list of safe tiles and returns the next step along the shortest path to the goal.
    Assumes that the world is a grid where each tile is connected to the adjacent tiles (not diagonally)
    Args:
        start (int, int): The coordinates of the start tile
        goal (int, int): The tile to find a shortest path towards
        safe_tiles [(int, int)]: A list of the tiles that can be traversed (should include start and goal)
    Returns ((int, int)/None): The next tile in the shortest path if one exists
    '''
    frontier = []
    heapq.heappush(frontier, (0, start))
    came_from = {}
    cost_so_far = {}
    came_from[start] = None
    cost_so_far[start] = 0
    
    # Run A* to get a dictionary containing for each tile, the previous tile along the path
    while len(frontier) > 0:
        distance, (x, y) = heapq.heappop(frontier)
        
        if (x,y) == goal:
            break
        
        for neighbor in [n for n in adjacent_tiles(x,y) if n in safe_tiles]:
            new_cost = cost_so_far[(x,y)] + 1 #Can be adapted to handle turning time
            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                cost_so_far[neighbor] = new_cost
                priority = new_cost + abs(x-neighbor[0]) + abs(y-neighbor[1])
                heapq.heappush(frontier, (priority, neighbor))
                came_from[neighbor] = (x,y)
    
    # Walk backward along the path until the first tile beyond the start is found
    # If the start tile is unreachable, return None
    current = goal
    while True:
        if current not in came_from:
            return None
        if came_from[current] == start:
            break
        current = came_from[current]
    
    return current

In [11]:
def play_game(agent, show=True):
    '''
    Initiates and runs a random Wumpus World by calling the Agent.act() function to get actions
    Args:
        agent (any): An object which has implemented an act function
        show (bool): Whether to print the states of the game or not
    '''
    w = WumpusWorld()
    while not w.game_over:
        if show:
            w.show()
        w.actuate(agent.act(w.perceive(), w.agent_pos, w.agent_dir))
    if show:
        w.show()

In [12]:
class SimpleAgent():
    '''
    Creates a simple knowledge-based agent for acting in the Wumpus World
    '''
    
    def __init__(self):
        pass
        # Initialize the agents internal parameters including the knowledge base
    
    def act(self, percepts, position, direction):
        '''
        Returns an action based on the agents percepts and position in this time step
            Args:
                percepts ([bool]): A list of, for each percept, whether it is active in this time step
                position (int, int): The coordinates of the tile the agent is currently in
                direction (int):  An int representing the direction the agent is currently facing
            Returns (str): A string representing the action to be taken (see the Wumpus World implementation for definitions)
        '''
        #Implement the earlier described procedure for a simple naive agent
        raise NotImplementedError('The SimpleAgent has not been implemented yet')

*Note*: The simple agent will not always find the gold. Sometimes the gold cannot be reached safely (for example when it is surrounded by pits, or when there is already a breeze or stench on the start tile), and the agent will then climb out without it. In some cases the agent might also end up looping forever.

In these cases it's fine to stop the cell and reset the simulation by running it again, or to test with a pre-set environment. Calling ```random.seed(...)``` with a fixed number before ```play_game``` gives the same world every time, e.g.:

```python
random.seed(3)
play_game(SimpleAgent())
```


In [13]:
# Execute this cell to play one game in the Wumpus World with the SimpleAgent

play_game(SimpleAgent())

	p		w	
			p	
p				
a		gp		
Performance: 0


NotImplementedError: The SimpleAgent has not been implemented yet